In [1]:
import os
import ot
import gc
import k3d
import torch
import trimesh
import warnings
from tqdm import *
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
from tqdm import tqdm
import seaborn as sns
from pathlib import Path
import scipy.sparse as sp
import matplotlib.cm as cm
from anndata import AnnData
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix
from numpy.random import RandomState
import matplotlib.font_manager as fm
from matplotlib.gridspec import GridSpec
from sklearn.metrics import jaccard_score
from scipy.stats import fisher_exact, norm
from sklearn.neighbors import NearestNeighbors
from typing import Literal, Optional, Tuple, Union
from matplotlib.colors import ListedColormap, rgb2hex
from sklearn.metrics.pairwise import euclidean_distances
from matplotlib.font_manager import fontManager, FontProperties
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar

import marsilea as ma
import re

In [4]:
import numpy as np
import pandas as pd
from matplotlib.colors import to_rgb, to_hex
import colorsys


def clamp(x, lo=0.0, hi=1.0):
    return max(lo, min(hi, x))


def tweak_color(
    hex_color,
    lighten=0.0,
    hue_shift=0.0,
    sat_scale=1.0,
):
    """
    在保留原始色系的基础上，调整：
    lighten: 向白色混合，0 原色，1 白色
    hue_shift: 色相偏移，建议小范围，如 -0.06 到 0.06
    sat_scale: 饱和度缩放，>1 更鲜艳，<1 更灰
    """
    r, g, b = to_rgb(hex_color)

    h, l, s = colorsys.rgb_to_hls(r, g, b)

    h = (h + hue_shift) % 1.0
    s = clamp(s * sat_scale)

    r2, g2, b2 = colorsys.hls_to_rgb(h, l, s)

    rgb = np.array([r2, g2, b2])
    white = np.array([1, 1, 1])

    new_rgb = rgb * (1 - lighten) + white * lighten
    return to_hex(new_rgb)


def symmetric_offsets(n, max_shift):
    """
    生成类似：
    0, +1, -1, +2, -2 ...
    这样最重要的 cluster 最接近原色，后面的逐渐偏移。
    """
    if n == 1 or max_shift == 0:
        return [0.0] * n

    offsets = [0.0]
    step = max_shift / max(1, np.ceil((n - 1) / 2))

    k = 1
    while len(offsets) < n:
        offsets.append(k * step)
        if len(offsets) < n:
            offsets.append(-k * step)
        k += 1

    return offsets[:n]


def build_leiden_palette_from_author(
    adata,
    leiden_key="cell_leiden",
    author_key="author_class",
    class_palette=None,
    min_lighten=0.0,
    max_lighten=0.45,
    max_hue_shift=0.07,
    min_sat_scale=0.75,
    max_sat_scale=1.15,
):
    obs = adata.obs[[leiden_key, author_key]].dropna().copy()

    obs[leiden_key] = obs[leiden_key].astype(str)
    obs[author_key] = obs[author_key].astype(str)

    confusion = pd.crosstab(
        obs[leiden_key],
        obs[author_key],
        normalize="index"
    )

    best_class = confusion.idxmax(axis=1)
    best_score = confusion.max(axis=1)

    mapping_df = pd.DataFrame({
        "cell_leiden": confusion.index,
        "matched_author_class": best_class.values,
        "matched_fraction": best_score.values,
    })

    leiden_palette = {}

    for author_class, sub_df in mapping_df.groupby("matched_author_class"):
        if class_palette is None or author_class not in class_palette:
            base_color = "#808080"
        else:
            base_color = class_palette[author_class]

        # 匹配度最高的 Leiden 最接近 author_class 原色
        sub_df = sub_df.sort_values("matched_fraction", ascending=False)

        n = len(sub_df)
        hue_offsets = symmetric_offsets(n, max_hue_shift)

        for rank, (_, row) in enumerate(sub_df.iterrows()):
            leiden = row["cell_leiden"]

            if n == 1:
                lighten = min_lighten
                sat_scale = 1.0
            else:
                t = rank / (n - 1)

                # 亮度逐渐增加
                lighten = min_lighten + (max_lighten - min_lighten) * t

                # 饱和度做轻微变化：前面的更鲜明，后面的略灰一些
                sat_scale = max_sat_scale + (min_sat_scale - max_sat_scale) * t

            leiden_palette[leiden] = tweak_color(
                base_color,
                lighten=lighten,
                hue_shift=hue_offsets[rank],
                sat_scale=sat_scale,
            )

    return leiden_palette, mapping_df, confusion

In [3]:
adata_path = "/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/rapids_analysis/merfish_mouseBrain_concat_embeddings.h5ad"
csv_path = "/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/example/05_merfish_mouseBrain/cluster_to_cluster_annotation_membership.csv"
json_path = "/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/rapids_analysis/author_colormaps/author_colormap.json"


adata = sc.read_h5ad(adata_path)
df = pd.read_csv(csv_path)

def strip_author_id(x):
    return re.sub(r"^\s*\d+\s+", "", str(x)).strip()

class_df = df[df["cluster_annotation_term_set_name"] == "class"].copy()
class_df["cluster_alias"] = class_df["cluster_alias"].astype(str)
class_df["author_class"] = class_df["cluster_annotation_term_name"].map(strip_author_id)

cluster_to_class = (
    class_df
    .drop_duplicates("cluster_alias")
    .set_index("cluster_alias")["author_class"]
    .to_dict()
)

adata.obs["author_class"] = (
    adata.obs["cluster_id_transfer"]
    .astype(str)
    .map(cluster_to_class)
    .astype("category")
)

import json

with open(json_path, "r", encoding="utf-8") as f:
    author_cmap = json.load(f)

print(author_cmap.keys())

class_palette = author_cmap["author_term_set_palettes"]["class"]
cluster_palette = author_cmap["obs_key_palettes"]["cluster_id_transfer"]
subclass_palette = author_cmap["obs_key_palettes"]["subclass_transfer"]

donor_id_colormap = {
    'C57BL6J-1': '#73BBF4', 
    'C57BL6J-2': '#284D76', 
    'C57BL6J-3': '#DF95D5', 
    'C57BL6J-4': '#791E25',
}
adata = adata[
    adata.obs["major_brain_region"].notna()
    & (adata.obs["major_brain_region"] != "n/a")
].copy()

dict_keys(['source_membership', 'source_h5ad', 'obs_key_palettes', 'author_term_set_palettes'])


In [12]:
subclass_palette

{'ABC NN': '#CC5CC1',
 'ACB-BST-FS D1 Gaba': '#FF4026',
 'AD Serpinb7 Glut': '#FF7383',
 'ADP-MPO Trp73 Glut': '#00CC6A',
 'AHN Onecut3 Gaba': '#553D66',
 'AHN-RCH-LHA Otp Fezf1 Glut': '#2E9957',
 'AHN-SBPV-PVHd Pdrm12 Gaba': '#BB7ACC',
 'APN C1ql2 Glut': '#5C64CC',
 'APN C1ql4 Glut': '#1FCC4D',
 'ARH-PVi Six6 Dopa-Gaba': '#6000FF',
 'ARH-PVp Tbx3 Gaba': '#CC3D76',
 'ARH-PVp Tbx3 Glut': '#CC4B1F',
 'AV Col27a1 Glut': '#16f2f2',
 'AVPV-MEPO-SFO Tbr1 Glut': '#1100CC',
 'Astro-CB NN': '#CC4E00',
 'Astro-NT NN': '#AD5CCC',
 'Astro-OLF NN': '#FF73CF',
 'Astro-TE NN': '#3DCCB1',
 'Astroependymal NN': '#0833ce',
 'B-PB Nr4a2 Glut': '#4ff7de',
 'BAM NN': '#66493D',
 'BST Tac2 Gaba': '#71FF4D',
 'BST-MPN Six3 Nrgn Gaba': '#996600',
 'BST-SI-AAA Six3 Slc22a3 Gaba': '#881FCC',
 'BST-po Iigp1 Glut': '#973DCC',
 'Bergmann NN': '#8E2600',
 'CA1-ProS Glut': '#0F3466',
 'CA2-FC-IG Glut': '#FF7391',
 'CA3 Glut': '#C973FF',
 'CB Granule Glut': '#FF4826',
 'CB PLI Gly-Gaba': '#009899',
 'CBN Dmbx1 Gaba':

In [5]:
cell_leiden_palette, leiden_author_map, cell_confusion = build_leiden_palette_from_author(
    adata,
    leiden_key="cell_leiden",
    author_key="subclass_transfer",
    class_palette=subclass_palette,
     min_lighten=0.20,
    max_lighten=0.50,
    max_hue_shift=2.00,
    min_sat_scale=0.50,
    max_sat_scale=100.00
)

In [6]:
import numpy as np
import pandas as pd
from matplotlib.colors import to_rgb, to_hex
import colorsys


def clamp(x, lo=0.0, hi=1.0):
    return max(lo, min(hi, x))


def tweak_color(
    hex_color,
    lighten=0.0,
    hue_shift=0.0,
    sat_scale=1.0,
):
    """
    在保留原始色系的基础上，调整：
    lighten: 向白色混合，0 原色，1 白色
    hue_shift: 色相偏移，建议小范围，如 -0.06 到 0.06
    sat_scale: 饱和度缩放，>1 更鲜艳，<1 更灰
    """
    r, g, b = to_rgb(hex_color)

    h, l, s = colorsys.rgb_to_hls(r, g, b)

    h = (h + hue_shift) % 1.0
    s = clamp(s * sat_scale)

    r2, g2, b2 = colorsys.hls_to_rgb(h, l, s)

    rgb = np.array([r2, g2, b2])
    white = np.array([1, 1, 1])

    new_rgb = rgb * (1 - lighten) + white * lighten
    return to_hex(new_rgb)


def symmetric_offsets(n, max_shift):
    """
    生成类似：
    0, +1, -1, +2, -2 ...
    这样最重要的 cluster 最接近原色，后面的逐渐偏移。
    """
    if n == 1 or max_shift == 0:
        return [0.0] * n

    offsets = [0.0]
    step = max_shift / max(1, np.ceil((n - 1) / 2))

    k = 1
    while len(offsets) < n:
        offsets.append(k * step)
        if len(offsets) < n:
            offsets.append(-k * step)
        k += 1

    return offsets[:n]


def build_leiden_palette_from_author(
    adata,
    leiden_key="cell_leiden",
    author_key="author_class",
    class_palette=None,
    min_lighten=0.0,
    max_lighten=0.45,
    max_hue_shift=0.07,
    min_sat_scale=0.75,
    max_sat_scale=1.15,
):
    obs = adata.obs[[leiden_key, author_key]].dropna().copy()

    obs[leiden_key] = obs[leiden_key].astype(str)
    obs[author_key] = obs[author_key].astype(str)

    confusion = pd.crosstab(
        obs[leiden_key],
        obs[author_key],
        normalize="index"
    )

    best_class = confusion.idxmax(axis=1)
    best_score = confusion.max(axis=1)

    mapping_df = pd.DataFrame({
        "cell_leiden": confusion.index,
        "matched_author_class": best_class.values,
        "matched_fraction": best_score.values,
    })

    leiden_palette = {}

    for author_class, sub_df in mapping_df.groupby("matched_author_class"):
        if class_palette is None or author_class not in class_palette:
            base_color = "#808080"
        else:
            base_color = class_palette[author_class]

        # 匹配度最高的 Leiden 最接近 author_class 原色
        sub_df = sub_df.sort_values("matched_fraction", ascending=False)

        n = len(sub_df)
        hue_offsets = symmetric_offsets(n, max_hue_shift)

        for rank, (_, row) in enumerate(sub_df.iterrows()):
            leiden = row["cell_leiden"]

            if n == 1:
                lighten = min_lighten
                sat_scale = 1.0
            else:
                t = rank / (n - 1)

                # 亮度逐渐增加
                lighten = min_lighten + (max_lighten - min_lighten) * t

                # 饱和度做轻微变化：前面的更鲜明，后面的略灰一些
                sat_scale = max_sat_scale + (min_sat_scale - max_sat_scale) * t

            leiden_palette[leiden] = tweak_color(
                base_color,
                lighten=lighten,
                hue_shift=hue_offsets[rank],
                sat_scale=sat_scale,
            )

    return leiden_palette, mapping_df, confusion

In [7]:
major_brain_region_colormap = {
    'Midbrain': '#FFA6FF',
    'Cerebellum': '#FFFDBC',
    'Isocortex': '#0D9F91',
    'Hippocampus': '#62178d',
    'Cortical_subplate': '#97EC93',
    'Medulla': '#FFA6FF',
    'Olfactory': '#A8ECD3',
    'Fiber_tracts': '#dcb67b',
    'Thalamus': '#FF909F',
    'Hypothalamus': '#F2483B',
    'Pallidum': '#B3C0DF',
    'Pons': '#FFA6FF',
    'Striatum': '#80C0E2',
    'Ventricular_systems': '#AAAAAA',
    'n/a': '#8fec63'
}

niche_leiden_palette, leiden_author_map, niche_confusion = build_leiden_palette_from_author(
    adata,
    leiden_key="niche_leiden",
    author_key="major_brain_region",
    class_palette=major_brain_region_colormap,
     min_lighten=0.20,
    max_lighten=0.50,
    max_hue_shift=2.00,
    min_sat_scale=0.50,
    max_sat_scale=100.00
)

In [8]:
import matplotlib.pyplot as plt

def add_umap_axis_arrow(
    ax,
    label_x="UMAP 1",
    label_y="UMAP 2",
    x_start=0.08,
    y_start=0.08,
    length=0.16,
    color="black",
    lw=1.2,
    fontsize=8,
    label_pad=0.015,
):
    ax.annotate(
        "",
        xy=(x_start + length, y_start),
        xytext=(x_start, y_start),
        xycoords="axes fraction",
        arrowprops=dict(
            arrowstyle="-|>",
            color=color,
            lw=lw,
            shrinkA=0,
            shrinkB=0,
            mutation_scale=10,
        ),
    )

    ax.annotate(
        "",
        xy=(x_start, y_start + length),
        xytext=(x_start, y_start),
        xycoords="axes fraction",
        arrowprops=dict(
            arrowstyle="-|>",
            color=color,
            lw=lw,
            shrinkA=0,
            shrinkB=0,
            mutation_scale=10,
        ),
    )

    ax.text(
        x_start + length / 2,
        y_start - label_pad,
        label_x,
        transform=ax.transAxes,
        ha="center",
        va="top",
        fontsize=fontsize,
        color=color,
    )

    ax.text(
        x_start - label_pad,
        y_start + length / 2,
        label_y,
        transform=ax.transAxes,
        ha="right",
        va="center",
        rotation=90,
        fontsize=fontsize,
        color=color,
    )

In [9]:
plot = sc.pl.embedding(
        adata, basis="X_umap_cell", color='donor_id',
        show=False, s=0.03, palette = donor_id_colormap, frameon = False, title = '',legend_loc  = None
    )
plot.set_aspect('equal')
add_umap_axis_arrow(
    plot,
    label_x="Cell UMAP1",
    label_y="Cell UMAP2",
    x_start=0.08,
    y_start=0.08,
    length=0.15,
    color="black",
    lw=1.2,
    fontsize=8,
    label_pad=0.006,
)

plt.savefig("/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/cell_uamp_donor_id.png", 
            bbox_inches="tight",
            transparent=True,
            facecolor='none',
            dpi = 450)
plt.close()

In [10]:


plot = sc.pl.embedding(
        adata, basis="X_umap_niche", color='donor_id',
        show=False, s=0.03, palette = donor_id_colormap, frameon = False, title = '',legend_loc  = None
    )
plot.set_aspect('equal')
add_umap_axis_arrow(
    plot,
    label_x="Niche UMAP1",
    label_y="Niche UMAP2",
    x_start=0.08,
    y_start=0.08,
    length=0.15,
    color="black",
    lw=1.2,
    fontsize=8,
    label_pad=0.006,
)
plt.savefig("/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/niche_uamp_donor_id.png", 
            bbox_inches="tight",
            transparent=True,
            facecolor='none',
            dpi = 450)
plt.close()

In [11]:
cell_leiden_dic = {
    'ob cell': ['335', '218', '79', '236', '265', '193', '187', '324', '162', '240', '66', '164', '183', '165', '248', '250', '300', '145', '81', '222', '29', '276', '40', '86', '185', '313', '234', '118', '123', '184'], # ob cell
    'ctx cell': ['171', '253', '133', '191', '312', '241', '172', '214', '247', '243', '199', '181', '232', '127', '245', '192', '284', '196', '295', '116', '154', '170', '112', '229', '318', '56', '37', '130', '319', '177'], # ctx cell
    'hip cell': ['311', '286', '168', '137', '139', '151', '242', '259', '296', '301', '150', '273', '131', '169', '128', '174', '94', '211', '281', '148', '261', '225', '230', '217', '119', '27', '251', '117', '244', '125'], # cnu cell
    'cb cell': ['341', '338', '278', '305', '216', '129', '326', '306', '330', '334', '96', '255', '287', '336', '310', '267', '317', '50', '333', '316', '322', '252', '190', '266', '213', '210', '161', '42', '14', '5'],
    'Fiber_tracts cell': ['100', '291', '282', '103', '95', '134', '272', '293', '277', '84', '197', '189', '141', '104', '36', '256', '210', '269', '207', '251', '140', '180', '331', '53', '47', '44', '98', '270', '204', '316'],
    'Midbrain cell': ['289', '329', '59', '285', '73', '158', '274', '205', '43', '140', '41', '132', '16', '35', '3'],
    'Thalamus cell': ['206', '320', '82', '297', '102', '186', '144', '90', '337', '257', '159', '205', '198', '16', '254', '121', '163', '35', '152', '209', '80', '43', '91', '132', '39', '221', '41', '26', '51', '12']
}
for celltype in cell_leiden_dic:
    # if os.path.exists(f"/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/cell_uamp_cell_leiden_{celltype}.png", ):
        # continue
    plot = sc.pl.embedding(
            adata, basis="X_umap_cell", color='cell_leiden',
            show=False, s=0.03, palette = cell_leiden_palette, frameon = False, title = '', groups = cell_leiden_dic[celltype],
        legend_loc  = None
        )
    plot.set_aspect('equal')
    add_umap_axis_arrow(
        plot,
        label_x="Cell UMAP1",
        label_y="Cell UMAP2",
        x_start=0.08,
        y_start=0.08,
        length=0.15,
        color="black",
        lw=1.2,
        fontsize=8,
        label_pad=0.006,
    )
    plt.savefig(f"/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/cell_uamp_cell_leiden_{celltype}.png", 
                bbox_inches="tight",
                transparent=True,
                facecolor='none',
                dpi = 450)
    plt.close()

In [11]:
author_class_dic = {
    'ob cell': ['OB Trdn Gaba', 'OB-mi Frmd7 Gaba', 'OB Eomes Ms4a15 Glut', 'LSX Sall3 Pax6 Gaba', 'OB-in Frmd7 Gaba', 'OB Meis2 Thsd7b Gaba', 'Astro-OLF NN', 'OB Dopa-Gaba', 'IT AON-TT-DP Glut', 'OB-out Frmd7 Gaba', 'L2/3 IT PIR-ENTl Glut', 'IA Mgp Gaba', 'PVHd-SBPV Six3 Prox1 Gaba', 'OB-STR-CTX Inh IMN', 'NDB-SI-ant Prdm12 Gaba', 'IT EP-CLA Glut', 'MEA-COA-BMA Ccdc42 Glut', 'PRC-PAG Tcf7l2 Irx2 Glut', 'STR-PAL Chst9 Gaba'], # ob cell
    'ctx cell': ['L5 IT CTX Glut', 'L4/5 IT CTX Glut', 'L5 ET CTX Glut', 'L2/3 IT CTX Glut', 'L5 NP CTX Glut', 'L6 IT CTX Glut', 'L2/3 IT RSP Glut', 'L6 CT CTX Glut', 'CLA-EPd-CTX Car3 Glut', 'L4 RSP-ACA Glut', 'L5/6 IT TPE-ENT Glut', 'Pvalb Gaba', 'Lamp5 Gaba', 'Vip Gaba', 'Sst Gaba', 'ABC NN', 'Sncg Gaba'], # ctx cell
    'hip cell': ['CA3 Glut', 'CA1-ProS Glut', 'DG Glut', 'CA2-FC-IG Glut', 'SUB-ProS Glut', 'NP SUB Glut', 'DG-PIR Ex IMN', 'L2 IT PPP-APr Glut', 'HPF CR Glut', 'RHP-COA Ndnf Gaba', 'L2/3 IT PPP Glut', 'Lamp5 Lhx6 Gaba', 'L2/3 IT ENT Glut', 'L6b/CT ENT Glut', 'Pvalb chandelier Gaba', 'Sncg Gaba', 'ENTmv-PA-COAp Glut', 'Astro-TE NN', 'CT SUB Glut', 'Sst Gaba'], # cnu cell
    'cb cell': ['CBX Purkinje Gaba', 'Bergmann NN', 'CBX MLI Megf11 Gaba', 'CBX MLI Cdh22 Gaba', 'CB PLI Gly-Gaba', 'CBX Golgi Gly-Gaba', 'CB Granule Glut', 'Astro-CB NN', 'CBN Dmbx1 Gaba', 'DCO UBC Glut', 'VCO Mafa Meis2 Glut', 'SPVI-SPVC Sall3 Lhx1 Gly-Gaba', 'Astroependymal NN', 'CBN Neurod2 Pvalb Glut'],
    'Fiber_tracts cell': ['NLL-po Pax7 Gaba', 'L6b CTX Glut', 'SPVC Ccdc172 Glut', 'OEC NN', 'PB Evx2 Glut', 'Astro-CB NN', 'NLL Gata3 Gly-Gaba', 'OB-STR-CTX Inh IMN', 'L6b/CT ENT Glut', 'Oligo NN', 'CBN Dmbx1 Gaba', 'SPVC Nmu Glut', 'STN-PSTN Pitx2 Glut', 'NLL-SOC Spp1 Glut', 'CT SUB Glut', 'PG-TRN-LRN Fat2 Glut'],
    'Midbrain cell': ['IC Tfap2d Maf Glut', 'SCsg Pde5a Glut', 'SC Tnnt1 Gli3 Gaba', 'SCsg Gabrr2 Gaba', 'PAG Pou4f1 Ebf2 Glut', 'IC Six3 En2 Gaba', 'PAG-MRN Tfap2b Glut', 'PAG Pou4f3 Glut', 'SCs Pax7 Nfia Gaba', 'CUN Evx2 Lhx2 Glut', 'PAG Pou4f1 Bnc2 Glut', 'PAG Pou4f2 Mesi2 Glut', 'LDT Vsx2 Nkx6-1 Nfib Glut', 'PAG-SC Pou4f1 Zic1 Glut', 'SC Bnc2 Glut', 'SCiw Pitx2 Glut', 'PAG Pou4f2 Glut', 'PAG-PPN Pax5 Sox21 Gaba', 'PAG-RN Nkx2-2 Otx1 Gaba', 'IPN-LDT Vsx2 Nkx6-1 Glut', 'SCs Dmbx1 Gaba', 'SC-PAG Lef1 Emx2 Gaba', 'SCig Foxb1 Glut', 'SCm-PAG Cdh23 Gaba', 'PAG-SC Neurod2 Meis2 Glut', 'MRN-PPN-CUN Pax8 Gaba', 'SPA-SPFm-SPFp-POL-PIL-PoT Sp9 Glut', 'MRN-VTN-PPN Pax5 Cdh23 Gaba', 'Astroependymal NN', 'PAG-MRN Pou3f1 Glut'],
    'Thalamus cell': ['VMH Fezf1 Glut', 'RE-Xi Nox4 Glut', 'VMH Nr5a1 Glut', 'CM-IAD-CL-PCN Sema5b Glut', 'ARH-PVp Tbx3 Glut', 'DMH-LHA Vgll2 Glut', 'PH-ant-LHA Otp Bsx Glut', 'ARH-PVp Tbx3 Gaba', 'TU-ARH Otp Six6 Gaba', 'TH Prkcd Grin2c Glut', 'PVHd-DMH Lhx6 Gaba', 'SBPV-PVa Six6 Satb2 Gaba', 'DMH Hmx2 Gaba', 'PVT-PT Ntrk1 Glut', 'PVHd-SBPV Six3 Prox1 Gaba', 'DMH-LHA Gsx1 Gaba', 'PVH-SO-PVa Otp Glut', 'LGv-SPFp-SPFm Nkx2-2 Tcf7l2 Gaba', 'Tanycyte NN', 'AHN-SBPV-PVHd Pdrm12 Gaba', 'RT-ZI Gnb3 Gaba', 'ZI Pax6 Gaba', 'SPA-SPFm-SPFp-POL-PIL-PoT Sp9 Glut', 'LH Pou4f1 Sox1 Glut', 'LGv-ZI Otx2 Gaba', 'PH-LHA Foxb1 Glut', 'MH Tac2 Glut', 'STN-PSTN Pitx2 Glut', 'Astro-NT NN', 'BST-MPN Six3 Nrgn Gaba'],
}

for celltype in author_class_dic:
    if os.path.exists(f"/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/cell_uamp_author_class_{celltype}.png", ):
        continue
    plot = sc.pl.embedding(
            adata, basis="X_umap_cell", color='subclass_transfer',
            show=False, s=0.03, palette = subclass_palette, frameon = False, title = '', groups = author_class_dic[celltype],
        legend_loc  = None
        )
    plot.set_aspect('equal')
    add_umap_axis_arrow(
        plot,
        label_x="Cell UMAP1",
        label_y="Cell UMAP2",
        x_start=0.08,
        y_start=0.08,
        length=0.15,
        color="black",
        lw=1.2,
        fontsize=8,
        label_pad=0.006,
    )
    plt.savefig(f"/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/cell_uamp_author_class_{celltype}.png", 
                bbox_inches="tight",
                transparent=True,
                facecolor='none',
                dpi = 450)
    plt.close()

In [12]:
niche_leiden_dic = {
    'ob cell': ['65', '67', '21', '47', '77', '19', '64', '10', '36', '69', '43'], # ob cell
    'ctx cell': ['23', '102', '91', '80', '68', '59', '56', '78', '74', '52', '20', '89', '29', '60', '61', '28', '48', '40'], # ctx cell
    'hip cell': ['90', '55', '45', '63', '54', '16', '0', '2', '41', '42'], # cnu cell
    'cb cell': ['57', '76', '32', '98', '94', '101', '46', '35', '26'],
    'Fiber_tracts cell': ['9', '24', '8', '46', '43', '86', '6', '22', '84', '69', '3', '66', '73', '48', '26', '104', '97'],
    'Midbrain cell': ['34', '39', '53', '3', '73'],
    'Thalamus cell': ['96', '51', '25', '70', '31', '50', '38', '71', '49', '4', '13', '1', '104'],
}
for niche in niche_leiden_dic:
    plot = sc.pl.embedding(
            adata, basis="X_umap_niche", color='niche_leiden',
            show=False, s=0.03, palette = niche_leiden_palette, frameon = False, title = '', groups = niche_leiden_dic[niche],
        legend_loc  = None
        )
    plot.set_aspect('equal')
    add_umap_axis_arrow(
        plot,
        label_x="Niche UMAP1",
        label_y="Niche UMAP2",
        x_start=0.08,
        y_start=0.08,
        length=0.15,
        color="black",
        lw=1.2,
        fontsize=8,
        label_pad=0.006,
    )
    plt.savefig(f"/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/cell_uamp_niche_leiden_{niche}.png", 
                bbox_inches="tight",
                transparent=True,
                facecolor='none',
                dpi = 450)
    plt.close()

In [13]:
set(adata.obs['major_brain_region'])

{'Cerebellum',
 'Cortical_subplate',
 'Fiber_tracts',
 'Hippocampus',
 'Hypothalamus',
 'Isocortex',
 'Medulla',
 'Midbrain',
 'Olfactory',
 'Pallidum',
 'Pons',
 'Striatum',
 'Thalamus',
 'Ventricular_systems'}

In [14]:
major_brain_regions = [
    'Olfactory', # ob cell
    'Isocortex', # ctx cell
    'Hippocampus', 
    'Cerebellum',
    'Fiber_tracts',
    'Midbrain',
    'Thalamus',
]
for major_brain_region in major_brain_regions:
    plot = sc.pl.embedding(
            adata, basis="X_umap_niche", color='major_brain_region',
            show=False, s=0.03, palette = major_brain_region_colormap, frameon = False, title = '', groups = major_brain_region,
        legend_loc  = None
        )
    plot.set_aspect('equal')
    add_umap_axis_arrow(
        plot,
        label_x="Niche UMAP1",
        label_y="Niche UMAP2",
        x_start=0.08,
        y_start=0.08,
        length=0.15,
        color="black",
        lw=1.2,
        fontsize=8,
        label_pad=0.006,
    )
    plt.savefig(f"/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/cell_uamp_major_brain_region_{major_brain_region}.png", 
                bbox_inches="tight",
                transparent=True,
                facecolor='none',
                dpi = 450)
    plt.close()

In [15]:
1

1

In [16]:
plot = sc.pl.embedding(
        adata, basis="X_umap_cell", color='cell_leiden',
        show=False, s=0.03, palette = cell_leiden_palette, frameon = False, title = '',
    legend_loc  = None
    )
plot.set_aspect('equal')
add_umap_axis_arrow(
    plot,
    label_x="Cell UMAP1",
    label_y="Cell UMAP2",
    x_start=0.08,
    y_start=0.08,
    length=0.15,
    color="black",
    lw=1.2,
    fontsize=8,
    label_pad=0.006,
)
plt.savefig("/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/cell_uamp_cell_leiden.png", 
            bbox_inches="tight",
            transparent=True,
            facecolor='none',
            dpi = 450)
plt.close()
# plt.show()

plot = sc.pl.embedding(
        adata, basis="X_umap_niche", color='niche_leiden',
        show=False, s=0.03, palette = niche_leiden_palette, frameon = False, title = '',
    legend_loc  = None
    )
plot.set_aspect('equal')
add_umap_axis_arrow(
    plot,
    label_x="Niche UMAP1",
    label_y="Cell UMAP2",
    x_start=0.08,
    y_start=0.08,
    length=0.15,
    color="black",
    lw=1.2,
    fontsize=8,
    label_pad=0.006,
)
plt.savefig("/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/niche_uamp_niche_leiden.png", 
            bbox_inches="tight",
            transparent=True,
            facecolor='none',
            dpi = 450)
plt.close()

In [17]:
major_brain_region_colormap = {'Midbrain': '#FFA6FF',
 'Cerebellum': '#FFFDBC',
 'Isocortex': '#0D9F91',
 'Hippocampus': '#62178d',
 'Cortical_subplate': '#97EC93',
 'Medulla': '#FFA6FF',
 'Olfactory': '#A8ECD3',
 'Fiber_tracts': '#dcb67b',
 'Thalamus': '#FF909F',
 'Hypothalamus': '#F2483B',
 'Pallidum': '#B3C0DF',
 'Pons': '#FFA6FF',
 'Striatum': '#80C0E2',
 'Ventricular_systems': '#AAAAAA',
 'n/a': '#8fec63'}

In [18]:
plot = sc.pl.embedding(
        adata, basis="X_umap_cell", color='subclass_transfer',
        show=False, s=0.03, palette = subclass_palette, frameon = False, title = '',
    legend_loc  = None
)
plot.set_aspect('equal')
add_umap_axis_arrow(
    plot,
    label_x="Cell UMAP1",
    label_y="Cell UMAP2",
    x_start=0.08,
    y_start=0.08,
    length=0.15,
    color="black",
    lw=1.2,
    fontsize=8,
    label_pad=0.006,
)
plt.savefig("/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/cell_uamp_author_class.png", 
            bbox_inches="tight",
            transparent=True,
            facecolor='none',
            dpi = 450)
plt.close()


plot = sc.pl.embedding(
        adata, basis="X_umap_niche", color='major_brain_region',
        show=False, s=0.03, palette = major_brain_region_colormap, frameon = False, title = '',
    legend_loc  = None
    )
plot.set_aspect('equal')
add_umap_axis_arrow(
    plot,
    label_x="Niche UMAP1",
    label_y="Niche UMAP2",
    x_start=0.08,
    y_start=0.08,
    length=0.15,
    color="black",
    lw=1.2,
    fontsize=8,
    label_pad=0.006,
)
plt.savefig("/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/niche_uamp_major_brain_region.png", 
            bbox_inches="tight",
            transparent=True,
            facecolor='none',
            dpi = 450)
plt.close()


In [19]:
import os
import numpy as np
import scanpy as sc
import k3d


def obs_3d_plot(
    adata_input,
    obs_key,
    spatial_key,
    colormap,
    output_html=None,
    groups=None,
    camera=None,
    point_size=80.0,
    opacity=0.85,
    scale=1.0,
    background_color=0xffffff,
    max_points_per_group=None,
    seed=0,
):
    """
    在 3D 空间中按照 adata.obs[obs_key] 绘制细胞类型 / 标签。

    Parameters
    ----------
    adata_input:
        AnnData 对象，或 h5ad 路径。
    obs_key:
        adata.obs 中的标签列，比如 "author_class", "cell_type", "subclass_transfer"。
    spatial_key:
        adata.obsm 中的 3D 坐标 key，比如 "ccf", "X_CCF", "X_spatial_coords"。
    colormap:
        dict，格式为 {"label": "#RRGGBB"}。
    output_html:
        如果提供路径，则保存 k3d html。
    groups:
        只绘制指定标签，比如 ["IT-ET Glut", "GABA"]。默认绘制全部。
    camera:
        k3d camera 参数。
    max_points_per_group:
        每个类别最多画多少点，用于大数据降采样。
    """

    if isinstance(adata_input, str):
        adata = sc.read_h5ad(adata_input)
    else:
        adata = adata_input

    if obs_key not in adata.obs:
        raise ValueError(f"adata.obs 中找不到 {obs_key}")

    if spatial_key not in adata.obsm:
        raise ValueError(f"adata.obsm 中找不到 {spatial_key}")

    coords = np.asarray(adata.obsm[spatial_key])
    if coords.shape[1] < 3:
        raise ValueError(f"adata.obsm['{spatial_key}'] 只有 {coords.shape[1]} 维，不能画 3D")

    positions = (coords[:, :3] * scale).astype(np.float32) * 9.8

    labels = adata.obs[obs_key].astype(str).to_numpy()

    if groups is None:
        if hasattr(adata.obs[obs_key], "cat"):
            categories = list(adata.obs[obs_key].cat.categories.astype(str))
        else:
            categories = sorted(pd.unique(labels))
    else:
        categories = [str(x) for x in groups]

    rng = np.random.default_rng(seed)

    plot = k3d.plot(
        name=f"3D {obs_key}",
        background_color=background_color,
        camera_auto_fit=False,
    )

    mesh = trimesh.load('/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/03_han_mouse_processed_mae_v1_train/average_template_10_sur.stl')
    
    plot = k3d.plot(name="Gene Expression 3D Plot",background_color=0xffffff,camera_auto_fit=False)
    
    stl_mesh = k3d.mesh(
        vertices=mesh.vertices.astype(np.float32),
        indices  =mesh.faces.astype(np.uint32),
        color    =0xB0B0B0,
        opacity  =0.20,
        wireframe=False,
        name='mouseMesh'
    )
    plot += stl_mesh
    
    for cat in categories:
        mask = labels == cat
        idx = np.where(mask)[0]

        if len(idx) == 0:
            print(f"跳过 {cat}: 没有细胞")
            continue

        if cat not in colormap:
            print(f"警告: {cat} 不在 colormap 中，使用灰色")
            color_hex = "#B0B0B0"
        else:
            color_hex = colormap[cat]

        if max_points_per_group is not None and len(idx) > max_points_per_group:
            idx = rng.choice(idx, size=max_points_per_group, replace=False)

        color_int = int(color_hex.lstrip("#"), 16)

        point_cloud = k3d.points(
            positions[idx],
            color=color_int,
            point_size=point_size,
            opacity=opacity,
            shader="3d",
            name=f"{cat} ({len(idx)})",
        )

        plot += point_cloud

    if camera is not None:
        plot.camera = camera

    plot.grid_visible = False
    plot.camera = [
        11485.284267462972,
        -14993.204389281993,
        127606.14254467492,
        69173.64700316277,
        57841.691822745124,
        43527.33431330532,
        0.21578089746207177,
        -0.8402988312227976,
        -0.4973293461440416]
    if output_html is not None:
        os.makedirs(os.path.dirname(output_html), exist_ok=True)
        with open(output_html, "w", encoding="utf-8") as f:
            f.write(plot.get_snapshot())

    return plot

In [20]:
plot = obs_3d_plot(
    adata,
    obs_key="subclass_transfer",
    spatial_key="X_CCF",
    colormap=subclass_palette,
    output_html="/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/test_author_class_3d.html",
    point_size=50,
    opacity=1,
    scale=1.0,
)

/home/share/huadjyin/home/zhoutao3/.conda/envs/stereotrack_2/lib/python3.10/site-packages/traittypes/traittypes.py:98: UserWarning: Given trait value dtype "float32" does not match required type "float32". A coerced copy has been created.
  warnings.warn(
/home/share/huadjyin/home/zhoutao3/.conda/envs/stereotrack_2/lib/python3.10/site-packages/traittypes/traittypes.py:98: UserWarning: Given trait value dtype "uint32" does not match required type "uint32". A coerced copy has been created.
  warnings.warn(


In [21]:
1

1

In [22]:
plot = obs_3d_plot(
    adata,
    obs_key="cell_leiden",
    spatial_key="X_CCF",
    colormap=cell_leiden_palette,
    output_html="/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/test_cell_leiden_3d.html",
    point_size=50,
    opacity=1,
    scale=1.0,
)

/home/share/huadjyin/home/zhoutao3/.conda/envs/stereotrack_2/lib/python3.10/site-packages/traittypes/traittypes.py:98: UserWarning: Given trait value dtype "float32" does not match required type "float32". A coerced copy has been created.
  warnings.warn(
/home/share/huadjyin/home/zhoutao3/.conda/envs/stereotrack_2/lib/python3.10/site-packages/traittypes/traittypes.py:98: UserWarning: Given trait value dtype "uint32" does not match required type "uint32". A coerced copy has been created.
  warnings.warn(


In [23]:
plot = obs_3d_plot(
    adata,
    obs_key="major_brain_region",
    spatial_key="X_CCF",
    colormap=major_brain_region_colormap,
    output_html="/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/test_major_brain_region_3d.html",
    point_size=50,
    opacity=1,
    scale=1.0,
)

/home/share/huadjyin/home/zhoutao3/.conda/envs/stereotrack_2/lib/python3.10/site-packages/traittypes/traittypes.py:98: UserWarning: Given trait value dtype "float32" does not match required type "float32". A coerced copy has been created.
  warnings.warn(
/home/share/huadjyin/home/zhoutao3/.conda/envs/stereotrack_2/lib/python3.10/site-packages/traittypes/traittypes.py:98: UserWarning: Given trait value dtype "uint32" does not match required type "uint32". A coerced copy has been created.
  warnings.warn(


In [24]:
plot = obs_3d_plot(
    adata,
    obs_key="niche_leiden",
    spatial_key="X_CCF",
    colormap=niche_leiden_palette,
    output_html="/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/test_niche_leiden_3d.html",
    point_size=50,
    opacity=1,
    scale=1.0,
)

/home/share/huadjyin/home/zhoutao3/.conda/envs/stereotrack_2/lib/python3.10/site-packages/traittypes/traittypes.py:98: UserWarning: Given trait value dtype "float32" does not match required type "float32". A coerced copy has been created.
  warnings.warn(
/home/share/huadjyin/home/zhoutao3/.conda/envs/stereotrack_2/lib/python3.10/site-packages/traittypes/traittypes.py:98: UserWarning: Given trait value dtype "uint32" does not match required type "uint32". A coerced copy has been created.
  warnings.warn(


In [25]:
import os
import re
import numpy as np
import pandas as pd
import scanpy as sc
import k3d
import trimesh


def obs_3d_plot_subset(
    adata_input,
    obs_key,
    spatial_key,
    colormap,
    output_html=None,
    groups=None,
    exclude_groups=None,
    group_regex=None,
    strict_groups=False,
    camera=None,
    point_size=80.0,
    opacity=0.85,
    scale=1.0,
    coord_scale=9.8,
    background_color=0xffffff,
    max_points_per_group=None,
    max_total_points=None,
    seed=0,
    mesh_path=None,
    mesh_color=0xB0B0B0,
    mesh_opacity=0.20,
    default_color="#B0B0B0",
    sort_groups=True,
    verbose=True,
):
    """
    在 3D 空间中按照 adata.obs[obs_key] 绘制指定细胞类型 / 标签。

    Parameters
    ----------
    adata_input:
        AnnData 对象，或 h5ad 路径。

    obs_key:
        adata.obs 中的标签列，比如 "author_class", "cell_type", "subclass_transfer"。

    spatial_key:
        adata.obsm 中的 3D 坐标 key，比如 "ccf", "X_CCF", "X_spatial_coords"。

    colormap:
        dict，格式为 {"label": "#RRGGBB"}。

    output_html:
        如果提供路径，则保存 k3d html。

    groups:
        只绘制指定标签。例如 ["IT-ET Glut", "GABA"]。
        如果为 None，则默认绘制全部。

    exclude_groups:
        排除指定标签。例如 ["Low quality", "Unknown"]。

    group_regex:
        只绘制匹配正则表达式的标签。例如 "Glut|GABA"。

    strict_groups:
        True 时，如果 groups 中有不存在的标签，直接报错。
        False 时，只给 warning 并跳过。

    max_points_per_group:
        每个类别最多绘制多少点。

    max_total_points:
        全图最多绘制多少点。会在筛选后的所有细胞中整体随机抽样。

    mesh_path:
        STL mesh 路径。如果为 None，则不绘制 mesh。

    coord_scale:
        额外坐标缩放系数。保留你原来代码里的 9.8。
    """

    if isinstance(adata_input, str):
        adata = sc.read_h5ad(adata_input)
    else:
        adata = adata_input

    if obs_key not in adata.obs:
        raise ValueError(f"adata.obs 中找不到 {obs_key}")

    if spatial_key not in adata.obsm:
        raise ValueError(f"adata.obsm 中找不到 {spatial_key}")

    coords = np.asarray(adata.obsm[spatial_key])
    if coords.ndim != 2 or coords.shape[1] < 3:
        raise ValueError(
            f"adata.obsm['{spatial_key}'] 的 shape 是 {coords.shape}，不能画 3D"
        )

    labels = adata.obs[obs_key].astype(str).to_numpy()
    all_categories = pd.Index(pd.unique(labels).astype(str))

    # ---------- decide categories ----------
    if groups is None:
        categories = all_categories.tolist()
    else:
        groups = [str(x) for x in groups]
        missing = [x for x in groups if x not in set(all_categories)]

        if missing and strict_groups:
            raise ValueError(
                f"groups 中有 {len(missing)} 个标签不在 adata.obs['{obs_key}'] 中: {missing}"
            )

        if missing and verbose:
            print(f"警告: 跳过不存在的 groups: {missing}")

        categories = [x for x in groups if x in set(all_categories)]

    if exclude_groups is not None:
        exclude_groups = set(str(x) for x in exclude_groups)
        categories = [x for x in categories if x not in exclude_groups]

    if group_regex is not None:
        pattern = re.compile(group_regex)
        categories = [x for x in categories if pattern.search(x)]

    if sort_groups and groups is None:
        categories = sorted(categories)

    if len(categories) == 0:
        raise ValueError("筛选后没有任何 category 可以绘制，请检查 groups/exclude_groups/group_regex")

    category_set = set(categories)

    # ---------- restrict to selected cells first ----------
    selected_mask = np.isin(labels, list(category_set))
    selected_idx = np.where(selected_mask)[0]

    if len(selected_idx) == 0:
        raise ValueError("筛选后没有任何细胞可以绘制")

    rng = np.random.default_rng(seed)

    if max_total_points is not None and len(selected_idx) > max_total_points:
        selected_idx = rng.choice(selected_idx, size=max_total_points, replace=False)
        selected_idx = np.sort(selected_idx)

    positions = (coords[:, :3] * scale * coord_scale).astype(np.float32)

    plot = k3d.plot(
        name=f"3D {obs_key}",
        background_color=background_color,
        camera_auto_fit=False,
    )

    # ---------- optional mesh ----------
    if mesh_path is not None:
        mesh = trimesh.load(mesh_path)

        stl_mesh = k3d.mesh(
            vertices=mesh.vertices.astype(np.float32),
            indices=mesh.faces.astype(np.uint32),
            color=mesh_color,
            opacity=mesh_opacity,
            wireframe=False,
            name="mesh",
        )

        plot += stl_mesh

    # ---------- points ----------
    selected_idx_set = set(selected_idx.tolist())

    for cat in categories:
        idx = np.where(labels == cat)[0]

        # 如果做过 max_total_points，需要限制在 selected_idx 里面
        idx = np.array([i for i in idx if i in selected_idx_set], dtype=int)

        if len(idx) == 0:
            if verbose:
                print(f"跳过 {cat}: 没有细胞")
            continue

        if max_points_per_group is not None and len(idx) > max_points_per_group:
            idx = rng.choice(idx, size=max_points_per_group, replace=False)

        color_hex = colormap.get(cat, default_color)

        if cat not in colormap and verbose:
            print(f"警告: {cat} 不在 colormap 中，使用默认灰色 {default_color}")

        color_int = int(color_hex.lstrip("#"), 16)

        point_cloud = k3d.points(
            positions[idx],
            color=color_int,
            point_size=point_size,
            opacity=opacity,
            shader="3d",
            name=f"{cat} ({len(idx)})",
        )

        plot += point_cloud

    if camera is not None:
        plot.camera = camera

    plot.grid_visible = False

    if output_html is not None:
        out_dir = os.path.dirname(output_html)
        if out_dir:
            os.makedirs(out_dir, exist_ok=True)

        with open(output_html, "w", encoding="utf-8") as f:
            f.write(plot.get_snapshot())

    return plot

In [26]:
camera = [
        11485.284267462972,
        -14993.204389281993,
        127606.14254467492,
        69173.64700316277,
        57841.691822745124,
        43527.33431330532,
        0.21578089746207177,
        -0.8402988312227976,
        -0.4973293461440416]

In [27]:

cell_leiden_dic = {
    'ob cell': ['335', '218', '79', '236', '265', '193', '187', '324', '162', '240', '66', '164', '183', '165', '248', '250', '300', '145', '81', '222', '29', '276', '40', '86', '185', '313', '234', '118', '123', '184'], # ob cell
    'ctx cell': ['171', '253', '133', '191', '312', '241', '172', '214', '247', '243', '199', '181', '232', '127', '245', '192', '284', '196', '295', '116', '154', '170', '112', '229', '318', '56', '37', '130', '319', '177'], # ctx cell
    'hip cell': ['311', '286', '168', '137', '139', '151', '242', '259', '296', '301', '150', '273', '131', '169', '128', '174', '94', '211', '281', '148', '261', '225', '230', '217', '119', '27', '251', '117', '244', '125'], # cnu cell
    'cb cell': ['341', '338', '278', '305', '216', '129', '326', '306', '330', '334', '96', '255', '287', '336', '310', '267', '317', '50', '333', '316', '322', '252', '190', '266', '213', '210', '161', '42', '14', '5'],
    'Fiber_tracts cell': ['100', '291', '282', '103', '95', '134', '272', '293', '277', '84', '197', '189', '141', '104', '36', '256', '210', '269', '207', '251', '140', '180', '331', '53', '47', '44', '98', '270', '204', '316'],
    'Midbrain cell': ['289', '329', '59', '285', '73', '158', '274', '205', '43', '140', '41', '132', '16', '35', '3'],
    'Thalamus cell': ['206', '320', '82', '297', '102', '186', '144', '90', '337', '257', '159', '205', '198', '16', '254', '121', '163', '35', '152', '209', '80', '43', '91', '132', '39', '221', '41', '26', '51', '12']
}
for cell_leiden in cell_leiden_dic:
    plot = obs_3d_plot_subset(
        adata,
        obs_key="cell_leiden",
        spatial_key="X_CCF",
        groups=cell_leiden_dic[cell_leiden],
        colormap=cell_leiden_palette,
        point_size=50,
        opacity=1,
        scale=1.0,
        mesh_path="/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/03_han_mouse_processed_mae_v1_train/average_template_10_sur.stl",
        output_html=f"/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/cell_leiden_{cell_leiden}.html",
        camera = camera
    )


/home/share/huadjyin/home/zhoutao3/.conda/envs/stereotrack_2/lib/python3.10/site-packages/traittypes/traittypes.py:98: UserWarning: Given trait value dtype "float32" does not match required type "float32". A coerced copy has been created.
  warnings.warn(
/home/share/huadjyin/home/zhoutao3/.conda/envs/stereotrack_2/lib/python3.10/site-packages/traittypes/traittypes.py:98: UserWarning: Given trait value dtype "uint32" does not match required type "uint32". A coerced copy has been created.
  warnings.warn(
/home/share/huadjyin/home/zhoutao3/.conda/envs/stereotrack_2/lib/python3.10/site-packages/traittypes/traittypes.py:98: UserWarning: Given trait value dtype "float32" does not match required type "float32". A coerced copy has been created.
  warnings.warn(
/home/share/huadjyin/home/zhoutao3/.conda/envs/stereotrack_2/lib/python3.10/site-packages/traittypes/traittypes.py:98: UserWarning: Given trait value dtype "uint32" does not match required type "uint32". A coerced copy has been create

In [28]:

niche_leiden_dic = {
    'ob cell': ['65', '67', '21', '47', '77', '19', '64', '10', '36', '69', '43'], # ob cell
    'ctx cell': ['23', '102', '91', '80', '68', '59', '56', '78', '74', '52', '20', '89', '29', '60', '61', '28', '48', '40'], # ctx cell
    'hip cell': ['90', '55', '45', '63', '54', '16', '0', '2', '41', '42'], # cnu cell
    'cb cell': ['57', '76', '32', '98', '94', '101', '46', '35', '26'],
    'Fiber_tracts cell': ['9', '24', '8', '46', '43', '86', '6', '22', '84', '69', '3', '66', '73', '48', '26', '104', '97'],
    'Midbrain cell': ['34', '39', '53', '3', '73'],
    'Thalamus cell': ['96', '51', '25', '70', '31', '50', '38', '71', '49', '4', '13', '1', '104'],
}

for niche_leiden in niche_leiden_dic:
    plot = obs_3d_plot_subset(
        adata,
        obs_key="niche_leiden",
        spatial_key="X_CCF",
        groups=niche_leiden_dic[niche_leiden],
        colormap=niche_leiden_palette,
        point_size=50,
        opacity=1,
        scale=1.0,
        mesh_path="/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/03_han_mouse_processed_mae_v1_train/average_template_10_sur.stl",
        output_html=f"/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/niche_leiden_{niche_leiden}.html",
        camera = camera
    )


/home/share/huadjyin/home/zhoutao3/.conda/envs/stereotrack_2/lib/python3.10/site-packages/traittypes/traittypes.py:98: UserWarning: Given trait value dtype "float32" does not match required type "float32". A coerced copy has been created.
  warnings.warn(
/home/share/huadjyin/home/zhoutao3/.conda/envs/stereotrack_2/lib/python3.10/site-packages/traittypes/traittypes.py:98: UserWarning: Given trait value dtype "uint32" does not match required type "uint32". A coerced copy has been created.
  warnings.warn(
/home/share/huadjyin/home/zhoutao3/.conda/envs/stereotrack_2/lib/python3.10/site-packages/traittypes/traittypes.py:98: UserWarning: Given trait value dtype "float32" does not match required type "float32". A coerced copy has been created.
  warnings.warn(
/home/share/huadjyin/home/zhoutao3/.conda/envs/stereotrack_2/lib/python3.10/site-packages/traittypes/traittypes.py:98: UserWarning: Given trait value dtype "uint32" does not match required type "uint32". A coerced copy has been create

In [29]:

author_class_dic = {
    'ob cell': ['OB Trdn Gaba', 'OB-mi Frmd7 Gaba', 'OB Eomes Ms4a15 Glut', 'LSX Sall3 Pax6 Gaba', 'OB-in Frmd7 Gaba', 'OB Meis2 Thsd7b Gaba', 'Astro-OLF NN', 'OB Dopa-Gaba', 'IT AON-TT-DP Glut', 'OB-out Frmd7 Gaba', 'L2/3 IT PIR-ENTl Glut', 'IA Mgp Gaba', 'PVHd-SBPV Six3 Prox1 Gaba', 'OB-STR-CTX Inh IMN', 'NDB-SI-ant Prdm12 Gaba', 'IT EP-CLA Glut', 'MEA-COA-BMA Ccdc42 Glut', 'PRC-PAG Tcf7l2 Irx2 Glut', 'STR-PAL Chst9 Gaba'], # ob cell
    'ctx cell': ['L5 IT CTX Glut', 'L4/5 IT CTX Glut', 'L5 ET CTX Glut', 'L2/3 IT CTX Glut', 'L5 NP CTX Glut', 'L6 IT CTX Glut', 'L2/3 IT RSP Glut', 'L6 CT CTX Glut', 'CLA-EPd-CTX Car3 Glut', 'L4 RSP-ACA Glut', 'L5/6 IT TPE-ENT Glut', 'Pvalb Gaba', 'Lamp5 Gaba', 'Vip Gaba', 'Sst Gaba', 'ABC NN', 'Sncg Gaba'], # ctx cell
    'hip cell': ['CA3 Glut', 'CA1-ProS Glut', 'DG Glut', 'CA2-FC-IG Glut', 'SUB-ProS Glut', 'NP SUB Glut', 'DG-PIR Ex IMN', 'L2 IT PPP-APr Glut', 'HPF CR Glut', 'RHP-COA Ndnf Gaba', 'L2/3 IT PPP Glut', 'Lamp5 Lhx6 Gaba', 'L2/3 IT ENT Glut', 'L6b/CT ENT Glut', 'Pvalb chandelier Gaba', 'Sncg Gaba', 'ENTmv-PA-COAp Glut', 'Astro-TE NN', 'CT SUB Glut', 'Sst Gaba'], # cnu cell
    'cb cell': ['CBX Purkinje Gaba', 'Bergmann NN', 'CBX MLI Megf11 Gaba', 'CBX MLI Cdh22 Gaba', 'CB PLI Gly-Gaba', 'CBX Golgi Gly-Gaba', 'CB Granule Glut', 'Astro-CB NN', 'CBN Dmbx1 Gaba', 'DCO UBC Glut', 'VCO Mafa Meis2 Glut', 'SPVI-SPVC Sall3 Lhx1 Gly-Gaba', 'Astroependymal NN', 'CBN Neurod2 Pvalb Glut'],
    'Fiber_tracts cell': ['NLL-po Pax7 Gaba', 'L6b CTX Glut', 'SPVC Ccdc172 Glut', 'OEC NN', 'PB Evx2 Glut', 'Astro-CB NN', 'NLL Gata3 Gly-Gaba', 'OB-STR-CTX Inh IMN', 'L6b/CT ENT Glut', 'Oligo NN', 'CBN Dmbx1 Gaba', 'SPVC Nmu Glut', 'STN-PSTN Pitx2 Glut', 'NLL-SOC Spp1 Glut', 'CT SUB Glut', 'PG-TRN-LRN Fat2 Glut'],
    'Midbrain cell': ['IC Tfap2d Maf Glut', 'SCsg Pde5a Glut', 'SC Tnnt1 Gli3 Gaba', 'SCsg Gabrr2 Gaba', 'PAG Pou4f1 Ebf2 Glut', 'IC Six3 En2 Gaba', 'PAG-MRN Tfap2b Glut', 'PAG Pou4f3 Glut', 'SCs Pax7 Nfia Gaba', 'CUN Evx2 Lhx2 Glut', 'PAG Pou4f1 Bnc2 Glut', 'PAG Pou4f2 Mesi2 Glut', 'LDT Vsx2 Nkx6-1 Nfib Glut', 'PAG-SC Pou4f1 Zic1 Glut', 'SC Bnc2 Glut', 'SCiw Pitx2 Glut', 'PAG Pou4f2 Glut', 'PAG-PPN Pax5 Sox21 Gaba', 'PAG-RN Nkx2-2 Otx1 Gaba', 'IPN-LDT Vsx2 Nkx6-1 Glut', 'SCs Dmbx1 Gaba', 'SC-PAG Lef1 Emx2 Gaba', 'SCig Foxb1 Glut', 'SCm-PAG Cdh23 Gaba', 'PAG-SC Neurod2 Meis2 Glut', 'MRN-PPN-CUN Pax8 Gaba', 'SPA-SPFm-SPFp-POL-PIL-PoT Sp9 Glut', 'MRN-VTN-PPN Pax5 Cdh23 Gaba', 'Astroependymal NN', 'PAG-MRN Pou3f1 Glut'],
    'Thalamus cell': ['VMH Fezf1 Glut', 'RE-Xi Nox4 Glut', 'VMH Nr5a1 Glut', 'CM-IAD-CL-PCN Sema5b Glut', 'ARH-PVp Tbx3 Glut', 'DMH-LHA Vgll2 Glut', 'PH-ant-LHA Otp Bsx Glut', 'ARH-PVp Tbx3 Gaba', 'TU-ARH Otp Six6 Gaba', 'TH Prkcd Grin2c Glut', 'PVHd-DMH Lhx6 Gaba', 'SBPV-PVa Six6 Satb2 Gaba', 'DMH Hmx2 Gaba', 'PVT-PT Ntrk1 Glut', 'PVHd-SBPV Six3 Prox1 Gaba', 'DMH-LHA Gsx1 Gaba', 'PVH-SO-PVa Otp Glut', 'LGv-SPFp-SPFm Nkx2-2 Tcf7l2 Gaba', 'Tanycyte NN', 'AHN-SBPV-PVHd Pdrm12 Gaba', 'RT-ZI Gnb3 Gaba', 'ZI Pax6 Gaba', 'SPA-SPFm-SPFp-POL-PIL-PoT Sp9 Glut', 'LH Pou4f1 Sox1 Glut', 'LGv-ZI Otx2 Gaba', 'PH-LHA Foxb1 Glut', 'MH Tac2 Glut', 'STN-PSTN Pitx2 Glut', 'Astro-NT NN', 'BST-MPN Six3 Nrgn Gaba'],
}

for celltype in author_class_dic:
    plot = obs_3d_plot_subset(
        adata,
        obs_key="subclass_transfer",
        spatial_key="X_CCF",
        groups=author_class_dic[celltype],
        colormap=subclass_palette,
        point_size=50,
        opacity=1,
        scale=1.0,
        mesh_path="/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/03_han_mouse_processed_mae_v1_train/average_template_10_sur.stl",
        output_html=f"/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/cell_author_{celltype}.html",
        camera = camera
    )


/home/share/huadjyin/home/zhoutao3/.conda/envs/stereotrack_2/lib/python3.10/site-packages/traittypes/traittypes.py:98: UserWarning: Given trait value dtype "float32" does not match required type "float32". A coerced copy has been created.
  warnings.warn(
/home/share/huadjyin/home/zhoutao3/.conda/envs/stereotrack_2/lib/python3.10/site-packages/traittypes/traittypes.py:98: UserWarning: Given trait value dtype "uint32" does not match required type "uint32". A coerced copy has been created.
  warnings.warn(
/home/share/huadjyin/home/zhoutao3/.conda/envs/stereotrack_2/lib/python3.10/site-packages/traittypes/traittypes.py:98: UserWarning: Given trait value dtype "float32" does not match required type "float32". A coerced copy has been created.
  warnings.warn(
/home/share/huadjyin/home/zhoutao3/.conda/envs/stereotrack_2/lib/python3.10/site-packages/traittypes/traittypes.py:98: UserWarning: Given trait value dtype "uint32" does not match required type "uint32". A coerced copy has been create